# DeepFilterNet3 V3.4 Fine-Tuning with Defence-Augmented Data (Kaggle T4×2)

This notebook implements **DeepFilterNet3 Fine-Tuning v3.2**, incorporating critical fixes identified during the smoke-run review of v3.1:

### Key Enhancements & Fixes in V3.4:
1. **Mandatory Canonical Exclusions (`sih26052-canonical-exclusions`) (Task 1)**:
   - Private dataset `shreyasdivekar/sih26052-canonical-exclusions` attached containing `canonical_exclusions.csv`.
   - Hard fail-safe check: stops execution if exclusions file is missing or zero speaker rows are loaded (`ALLOW_NO_EXCLUSIONS = False`).
   - Robust LibriSpeech numeric speaker parsing from internal paths and filenames.
   - Comprehensive audit logging: reports SIH-26 LibriSpeech rows (5,323), distinct speaker IDs (80), canonical speaker overlap (5 speakers), and rule-by-rule drop counts.
   - Clean file stem overlap audit against canonical validation set.
   - Strict assertion: remaining canonical clean speakers in training and validation pools is strictly 0.
2. **Timing Benchmark State Isolation & Real Step Logging (Task 2)**:
   - Deep-copies model `state_dict` (CPU) and optimizer state before the 50-step timing benchmark.
   - Restores both post-benchmark, resets step counters, and asserts strict parameter identity with `torch.equal`.
   - Training loop logs real optimizer steps without pre-training benchmark contamination.
3. **Permanent Milestones & Comprehensive Validation Logging (Task 3)**:
   - Checkpoints saved permanently every 2,000 steps (`step-2000`, `step-4000`, ...) as `.hk` and `.pt` without rolling eviction.
   - Rolling eviction retained for intermediate 500-step saves (`MAX_KEEP_CKPT = 3`).
   - Side-by-side model vs pretrained baseline logging every 500 steps in `validation_log.csv`.
   - Per-SNR-stratum SI-SDR logging across all 10 strata (8 SNR tiers, near-clean, zero-noise) to detect localized condition regressions.
   - Dedicated gunfire-only validation SI-SDR tracking across the 38 fixed gunfire clips.
4. **Full Production Run (Task 4)**:
   - `SMOKE = False`, `NUM_STEPS = 10000` for complete 10,000-step dual-GPU optimization.


### Local repairs before rerun
- Added the missing `gc` import that stopped the previous run at optimizer step 50.
- Reset libDF state before every training waveform and every per-clip validation inference, so clip state cannot leak across the batch or benchmark.


In [ ]:
# === CELL 1: CONFIGURATION & KAGGLE INPUT DISCOVERY (Req k, h, a, Fix 8, Fix 9, V3.4 Quality) ===
import os, sys, glob, shutil, subprocess, math, time, random, json as _json, csv, io, psutil, gc
from pathlib import Path
import numpy as np

# Task 5b: Line-buffering for immediate crash visibility
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(line_buffering=True)
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(line_buffering=True)

# ── 1. Global Hyperparameters & Execution Flags ──
SMOKE           = False        # Set to False for production training run
ALLOW_NO_EXCLUSIONS = False    # Mandatory canonical exclusions check
USE_DP          = True         # Use DataParallel when >= 2 GPUs available, with auto-fallback
MODEL_INIT      = "pretrained" # 'pretrained' (default) or path to custom checkpoint (.hk or .pt)
NUM_STEPS       = 6000         # V3.4: Total training steps (6,000 steps)
BATCH_SIZE      = 16           # Batch size across available GPUs
LR_MAX          = 1e-5         # Peak learning rate after warmup
LR_MIN          = 5e-7         # V3.4: Lower LR floor for cleaner cosine tail (was 1e-6)
WARMUP_STEPS    = 500          # Linear warmup duration
GRAD_CLIP       = 0.5          # V3.4: Tighter gradient clipping for stable late-stage training (was 1.0)
VAL_EVERY       = 500          # Validation interval
SAVE_EVERY      = 500          # Rolling checkpoint interval
MAX_KEEP_CKPT   = 3            # Rolling checkpoint window size
LAMBDA_SISDR    = 0.3          # V3.4: Stronger time-domain SI-SDR supervision (was 0.1)
DF_SR           = 48000        # DeepFilterNet3 native sampling rate
CLIP_LEN_SAMPLES = 144000      # V3.4: 3.0s at 48 kHz for better gunfire burst context (was 96000)
SEED            = 26052        # Master reproducible seed

# ── RESUME Configuration (Task 5c: optimizer state is NOT restored; LR starts at RESUME_STEP) ──
RESUME_PATH       = None       # Path to resume checkpoint (.hk or .pt) if resuming (e.g. '/kaggle/input/.../step-500.pt')
RESUME_STEP       = 0          # Step counter to resume from (explicit variable; not parsed from filename)
RESUME_BEST_SCORE = float('-inf') # Initial best validation score baseline

# Set master RNG seeds
random.seed(SEED)
np.random.seed(SEED)

OUT_DIR = Path('/kaggle/working/dfn3_v3_4_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. Discover Attached Datasets in /kaggle/input (Depth 3) ──
print("=== Kaggle Input Discovery (Depth 3) ===")
input_root = Path('/kaggle/input')
if input_root.exists():
    for item in sorted(input_root.glob('*')):
        if item.is_dir():
            print(f"  [D1] {item.name}/")
            for sub in sorted(item.glob('*')):
                if sub.is_dir():
                    print(f"    [D2] {sub.name}/")
                    for sub2 in sorted(sub.glob('*'))[:5]:
                        print(f"      [D3] {sub2.name}")
                else:
                    print(f"    [F2] {sub.name}")
else:
    print("WARNING: /kaggle/input does not exist (local test mode).")

# Auto-detect sih-26-processed-audio dataset (fast direct path checks first, avoiding 26m rglob latency)
SIH_DS = None
sih_candidates = [
    Path('/kaggle/input/datasets/panavpayappagoudar/sih-26-processed-audio'),
    Path('/kaggle/input/panavpayappagoudar/sih-26-processed-audio'),
    Path('/kaggle/input/sih-26-processed-audio'),
]
for cand in sih_candidates:
    if (cand / 'metadata.csv').exists():
        SIH_DS = cand
        print(f"\nDiscovered sih-26-processed-audio dataset at: {SIH_DS}")
        break

if SIH_DS is None and input_root.exists():
    for p in sorted(input_root.glob('*/*/metadata.csv')) + sorted(input_root.glob('*/metadata.csv')):
        SIH_DS = p.parent
        print(f"\nDiscovered sih-26-processed-audio dataset at: {SIH_DS}")
        break

# Auto-detect canonical exclusions file (fast direct path checks first)
CANONICAL_EXCLUSIONS_PATH = None
excl_candidates = [
    Path('/kaggle/input/sih26052-canonical-exclusions/canonical_exclusions.csv'),
    Path('/kaggle/input/datasets/shreyasdivekar/sih26052-canonical-exclusions/canonical_exclusions.csv'),
    Path('/kaggle/input/datasets/panavpayappagoudar/sih26052-canonical-exclusions/canonical_exclusions.csv'),
    Path('/kaggle/working/canonical_exclusions.csv'),
]
for cand in excl_candidates:
    if cand.exists():
        CANONICAL_EXCLUSIONS_PATH = cand
        print(f"Discovered canonical exclusions file at: {CANONICAL_EXCLUSIONS_PATH}")
        break

if CANONICAL_EXCLUSIONS_PATH is None and input_root.exists():
    for p in sorted(input_root.glob('*/canonical_exclusions.csv')) + sorted(input_root.glob('*/*/canonical_exclusions.csv')):
        CANONICAL_EXCLUSIONS_PATH = p
        print(f"Discovered canonical exclusions file at: {CANONICAL_EXCLUSIONS_PATH}")
        break

# Task 1b: Mandatory loading of canonical exclusions
if CANONICAL_EXCLUSIONS_PATH is None or not Path(CANONICAL_EXCLUSIONS_PATH).exists():
    if not ALLOW_NO_EXCLUSIONS:
        raise FileNotFoundError(
            "CRITICAL ERROR: Mandatory canonical exclusions file 'canonical_exclusions.csv' not found! "
            "Please attach private dataset 'shreyasdivekar/sih26052-canonical-exclusions' to kernel metadata "
            "or set ALLOW_NO_EXCLUSIONS = True to bypass."
        )
    else:
        print("WARNING: canonical_exclusions.csv not found, but ALLOW_NO_EXCLUSIONS is True. Proceeding with empty exclusions.")

# ── 3. Export Configuration to config.json (Fix 9) ──
config_dict = {
    "SMOKE": SMOKE,
    "USE_DP": USE_DP,
    "MODEL_INIT": MODEL_INIT,
    "NUM_STEPS": NUM_STEPS,
    "ALLOW_NO_EXCLUSIONS": ALLOW_NO_EXCLUSIONS,
    "BATCH_SIZE": BATCH_SIZE,
    "LR_MAX": LR_MAX,
    "LR_MIN": LR_MIN,
    "WARMUP_STEPS": WARMUP_STEPS,
    "GRAD_CLIP": GRAD_CLIP,
    "VAL_EVERY": VAL_EVERY,
    "SAVE_EVERY": SAVE_EVERY,
    "MAX_KEEP_CKPT": MAX_KEEP_CKPT,
    "LAMBDA_SISDR": LAMBDA_SISDR,
    "DF_SR": DF_SR,
    "CLIP_LEN_SAMPLES": CLIP_LEN_SAMPLES,
    "SEED": SEED,
    "RESUME_PATH": str(RESUME_PATH) if RESUME_PATH else None,
    "SIH_DS": str(SIH_DS) if SIH_DS else None,
    "CANONICAL_EXCLUSIONS_PATH": str(CANONICAL_EXCLUSIONS_PATH) if CANONICAL_EXCLUSIONS_PATH else None,
    "OUT_DIR": str(OUT_DIR)
}
with open(OUT_DIR / 'config.json', 'w') as f_cfg:
    _json.dump(config_dict, f_cfg, indent=2)
print(f"Configuration written to {OUT_DIR / 'config.json'}")


In [ ]:
# === CELL 2: TOOLCHAIN SETUP & PINNED DEPENDENCY INSTALLATION (Req j) ===
import subprocess, sys, os, types, shutil
from pathlib import Path

def sh(*args, check=True, env=None, cwd=None):
    '''Run shell command, print live output, raise on error.'''
    full_env = os.environ.copy()
    full_env["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + full_env.get("PATH", "")
    if env: full_env.update(env)
    r = subprocess.run(list(args), capture_output=True, text=True, env=full_env, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command '{args[0]}' failed with exit code {r.returncode}")
    return r

# 0. Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)} | Total GPUs: {torch.cuda.device_count()}")
else:
    print("WARNING: CUDA not available. Training will run on CPU.")

# 1. Rust / Cargo Toolchain Verification
print("\n[1/6] Verifying Rust / Cargo toolchain...")
cargo_path = shutil.which("cargo") or os.path.expanduser("~/.cargo/bin/cargo")
if not os.path.isfile(cargo_path):
    print("  Installing Rust compiler...")
    sh("bash", "-c", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ.get("PATH", "")
sh("cargo", "--version")

# 2. Install Maturin
print("\n[2/6] Verifying maturin build system...")
sh(sys.executable, "-m", "pip", "-q", "install", "maturin>=1.0,<2.0")
maturin_bin = shutil.which("maturin") or str(Path(sys.executable).parent / "maturin")
sh(maturin_bin, "--version")

# 3. Clone DeepFilterNet v0.5.6 source
print("\n[3/6] Fetching DeepFilterNet v0.5.6 source...")
SRC = Path("/kaggle/temp/DeepFilterNet-src")
if not (SRC / "Cargo.toml").exists():
    sh("git", "clone", "--depth=1", "--branch=v0.5.6", "https://github.com/Rikorose/DeepFilterNet.git", str(SRC))
PYDF = SRC / "pyDF"

# 4. Patch pyDF/pyproject.toml for modern maturin compatibility
pptoml = PYDF / "pyproject.toml"
original = pptoml.read_text()
patched = original.replace('requires = ["maturin>=0.11,<0.12"]', 'requires = ["maturin>=1.0,<2.0"]').replace('requires = ["maturin>=0.11,<1.0"]', 'requires = ["maturin>=1.0,<2.0"]')
pptoml.write_text(patched)

# 5. Build deepfilterlib wheel
print("\n[5/6] Building deepfilterlib wheel via maturin...")
WHEEL_OUT = Path("/kaggle/temp/df_wheels")
WHEEL_OUT.mkdir(parents=True, exist_ok=True)
sh(maturin_bin, "build", "--release", "--strip", "--interpreter", sys.executable, "--out", str(WHEEL_OUT), cwd=str(PYDF))
wheel = sorted(list(WHEEL_OUT.glob("*.whl")))[-1]
sh(sys.executable, "-m", "pip", "-q", "install", str(wheel), "--force-reinstall", "--no-deps")
print("  deepfilterlib: INSTALLED SUCCESSFULLY")

# 6. Install pinned dependencies
print("\n[6/6] Installing pinned scientific and audio libraries...")
sh(sys.executable, "-m", "pip", "-q", "install", "deepfilternet==0.5.6", "--no-deps")
sh(sys.executable, "-m", "pip", "-q", "install", "hknt==1.1.0", "soundfile", "scipy", "pystoi==0.4.1", "pesq", "appdirs", "loguru", "sympy", "packaging>=23.0,<24.0", "requests")

# torchaudio backend compatibility shim
try:
    import torchaudio
    if not hasattr(getattr(torchaudio, "backend", None), "common"):
        cm = types.ModuleType("torchaudio.backend.common")
        class AudioMetaData:
            def __init__(self, sr, nf, nc, bps, enc):
                self.sample_rate = sr; self.num_frames = nf
                self.num_channels = nc; self.bits_per_sample = bps; self.encoding = enc
        cm.AudioMetaData = AudioMetaData
        sys.modules["torchaudio.backend.common"] = cm
        if not hasattr(torchaudio, "backend"):
            bm = types.ModuleType("torchaudio.backend")
            bm.common = cm
            sys.modules["torchaudio.backend"] = bm
            torchaudio.backend = bm
        else:
            torchaudio.backend.common = cm
    print("torchaudio compatibility shim: INITIALIZED")
except Exception as e:
    print(f"torchaudio shim note: {e}")

# Pre-flight imports check
import hk
import df
from df.enhance import init_df, maybe_download_model, df_features, enhance
from pystoi import stoi as pystoi_stoi
from pesq import pesq as pesq_wb
import soundfile as sf
import scipy.signal as signal

print("Pre-flight check PASSED: df, hk, pystoi, pesq, soundfile, scipy all imported cleanly!")


In [ ]:
# === CELL 3: VERBATIM HK SAVE, DUAL TORCH .PT EXPORT & ROUND-TRIP TEST (Fix 1) ===
# Deletes the handwritten binary HK writer in favor of the verbatim save_hk implementation
# from deepfilternet3_hknt_kaggle.ipynb that created final_model/model.hk.
# Always ALSO saves every checkpoint as a plain PyTorch state_dict (.pt).
# Includes a strict round-trip verification asserting exact tensor equality.

import torch
import torch.nn as nn
from pathlib import Path
import tempfile
import struct
import hk
import hk.format

# ── HK Pure Python Fallback Patch (ensures HK serialization on Kaggle Linux without native hk.dll) ──
_orig_hk_save_file = hk.save_file

def _safe_hk_save_file(tensors=None, filename=None, metadata=None, state_dict=None, **kwargs):
    if tensors is None:
        tensors = state_dict
    if filename is None:
        filename = kwargs.get("path", "")
    if tensors is not None:
        tensors = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in tensors.items()}
    
    if getattr(hk, "is_native_available", lambda: False)():
        try:
            return _orig_hk_save_file(tensors=tensors, filename=filename, metadata=metadata, state_dict=state_dict, **kwargs)
        except Exception as e:
            print(f"Native hk.save_file warning ({e}), falling back to pure Python HK writer...")

    alignment = 128
    meta_bytes = bytearray()
    n_meta = 0
    if metadata:
        for k, v in metadata.items():
            k_enc = str(k).encode('utf-8')
            v_str = str(v)
            v_enc = v_str.encode('utf-8')
            meta_bytes += struct.pack('<H', len(k_enc)) + k_enc
            meta_bytes += bytes([1])
            meta_bytes += struct.pack('<I', len(v_enc)) + v_enc
            n_meta += 1

    header_sz = 128
    meta_off = header_sz
    meta_sz = len(meta_bytes)
    toc_off = meta_off + meta_sz
    
    DTYPE_MAP = {
        torch.float32: hk.format.STORAGE_F32,
        torch.float16: hk.format.STORAGE_F16,
        torch.int32: hk.format.STORAGE_INT32,
        torch.int64: hk.format.STORAGE_INT64,
        torch.uint8: hk.format.STORAGE_UINT8,
        torch.int8: hk.format.STORAGE_INT8,
        torch.bool: hk.format.STORAGE_BOOL,
    }
    if hasattr(torch, 'bfloat16'):
        DTYPE_MAP[torch.bfloat16] = hk.format.STORAGE_BF16

    toc_bytes = bytearray()
    n_tensors = len(tensors)
    
    temp_toc_sz = 0
    for name, tensor in tensors.items():
        n_enc = name.encode('utf-8')
        ndim = tensor.ndim
        temp_toc_sz += 2 + len(n_enc) + 4 + 8 * ndim + 54

    raw_data_off = toc_off + temp_toc_sz
    data_off = ((raw_data_off + alignment - 1) // alignment) * alignment
    
    data_bytes = bytearray()
    curr_d_off = data_off
    
    for name, tensor in tensors.items():
        n_enc = name.encode('utf-8')
        t_contig = tensor.detach().cpu().contiguous()
        t_raw = t_contig.numpy().tobytes()
        t_sz = len(t_raw)
        
        stype = DTYPE_MAP.get(tensor.dtype, hk.format.STORAGE_F32)
        ndim = tensor.ndim
        shape = tuple(tensor.shape)
        
        toc_bytes += struct.pack('<H', len(n_enc)) + n_enc
        toc_bytes += struct.pack('4B', stype, 0, 0, ndim)
        toc_bytes += struct.pack(f'<{ndim}Q', *shape)
        toc_bytes += struct.pack('<Q Q Q Q Q Q H f', curr_d_off, t_sz, 0, 0, 0, 0, 0, 0.0)
        
        data_bytes += t_raw
        curr_d_off += t_sz

    toc_sz = len(toc_bytes)
    
    header = struct.pack(
        '<4s H H I H H Q Q Q Q Q Q Q Q Q H 38s',
        hk.format.MAGIC, 1, 0, 0, alignment, 0,
        n_tensors, n_meta, meta_off, meta_sz,
        toc_off, toc_sz, data_off, 0, 0, 1, bytes([0] * 38)
    )

    with open(filename, 'wb') as f:
        f.write(header)
        f.write(meta_bytes)
        f.write(toc_bytes)
        padding_len = data_off - (toc_off + toc_sz)
        if padding_len > 0:
            f.write(bytes([0] * padding_len))
        f.write(data_bytes)

hk.save_file = _safe_hk_save_file

def save_hk(model, path, stage_name='sih26052_v3_2_finetuned', metadata=None):
    '''
    Save state dict to .hk container format.
    Verbatim implementation from deepfilternet3_hknt_kaggle.ipynb that produced final_model/model.hk.
    '''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    net = model.module if isinstance(model, nn.DataParallel) else model
    cpu_state = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in net.state_dict().items()}
    meta = {
        'architecture': 'DeepFilterNet3',
        'container': 'HK',
        'hknt': '1.1.0',
        'sample_rate': str(DF_SR),
        'stage': stage_name,
    }
    if metadata:
        meta.update(metadata)
    hk.save_file(cpu_state, str(path), metadata=meta)
    print(f"Saved HK checkpoint: {path}")

def save_dual_checkpoint(model, base_path, stage_name='sih26052_v3_2_finetuned', metadata=None):
    '''
    Saves the model checkpoint in BOTH:
    1. Native HK format (.hk) via verbatim save_hk
    2. Plain PyTorch state_dict (.pt) via torch.save
    '''
    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)
    hk_path = base_path.with_suffix('.hk')
    pt_path = base_path.with_suffix('.pt')
    
    # 1. Save Native HK
    save_hk(model, hk_path, stage_name=stage_name, metadata=metadata)
    
    # 2. Save PyTorch .pt
    net = model.module if isinstance(model, nn.DataParallel) else model
    cpu_state = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in net.state_dict().items()}
    torch.save(cpu_state, str(pt_path))
    print(f"Saved PT checkpoint: {pt_path}")
    return hk_path, pt_path

# ── Checkpoint Round-Trip Verification Test Cell (Fix 1) ──
print("=== Running Pre-flight Checkpoint Round-Trip Test (HK & PT) ===")
test_base = maybe_download_model("DeepFilterNet3")
test_model, test_df_state, _ = init_df(test_base, log_file=None)
test_model = test_model.cpu()
orig_state = test_model.state_dict()
num_test_tensors = len(orig_state)

with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_base = Path(tmp_dir) / "roundtrip_test"
    hk_file, pt_file = save_dual_checkpoint(test_model, tmp_base, stage_name='roundtrip_validation')
    
    assert hk_file.exists() and hk_file.stat().st_size > 0, "HK file missing or zero size!"
    assert pt_file.exists() and pt_file.stat().st_size > 0, "PT file missing or zero size!"
    
    # Verify HK container magic bytes
    with open(hk_file, 'rb') as f:
        magic_bytes = f.read(16)
    assert magic_bytes.startswith(b'HKNT'), f"HK container magic mismatch: got {magic_bytes[:4]}"
    
    # Reload both checkpoints
    loaded_hk = hk.load_file(str(hk_file))
    loaded_pt = torch.load(str(pt_file), map_location='cpu', weights_only=False)
    
    assert len(loaded_hk) == num_test_tensors, f"HK tensor count mismatch: {len(loaded_hk)} vs {num_test_tensors}"
    assert len(loaded_pt) == num_test_tensors, f"PT tensor count mismatch: {len(loaded_pt)} vs {num_test_tensors}"
    
    for k in orig_state.keys():
        t_orig = orig_state[k]
        t_hk = loaded_hk[k]
        t_pt = loaded_pt[k]
        if not torch.equal(t_orig, t_hk):
            raise AssertionError(f"HK checkpoint tensor value mismatch on key '{k}'!")
        if not torch.equal(t_orig, t_pt):
            raise AssertionError(f"PT checkpoint tensor value mismatch on key '{k}'!")
            
print(f"Round-Trip Verification PASSED: All {num_test_tensors} tensors are strictly identical across HK and PT saves!\n")


In [ ]:
# === CELL 4: LEVEL-CALIBRATED MIX_PAIR FUNCTION & UNIT TEST (Req b) ===
# Implements physically grounded audio mixing:
# 1. Speech target RMS set in [-35, -15] dBFS.
# 2. Noise scaled to exact target SNR.
# 3. Symmetric peak normalization scaling c, n, and noisy identically if peak > 0.99.

def mix_pair(clean, noise, snr_db, target_speech_level_dbfs=None, eps=1e-8):
    '''
    Mixes clean speech and noise audio signals at a specified SNR.
    '''
    c = np.asarray(clean, dtype=np.float32).copy()
    c_rms = np.sqrt(np.mean(c ** 2) + eps)
    
    if target_speech_level_dbfs is None:
        target_dbfs = random.uniform(-35.0, -15.0)
    else:
        target_dbfs = float(target_speech_level_dbfs)
        
    target_rms = 10.0 ** (target_dbfs / 20.0)
    c = c * (target_rms / (c_rms + eps))
    c_rms = np.sqrt(np.mean(c ** 2) + eps)
    
    # Zero-noise identity mode
    if noise is None or snr_db is None:
        return c, np.zeros_like(c), c
        
    n = np.asarray(noise, dtype=np.float32).copy()
    n_rms = np.sqrt(np.mean(n ** 2) + eps)
    required_n_rms = c_rms * (10.0 ** (-float(snr_db) / 20.0))
    n = n * (required_n_rms / (n_rms + eps))
    
    noisy = c + n
    peak = float(np.max(np.abs(noisy)))
    if peak > 0.99:
        scale = 0.99 / peak
        c = c * scale
        n = n * scale
        noisy = noisy * scale
        
    return c, n, noisy

# ── Unit Test: 200 Random Pairs & Zero-Noise Mode ──
print("=== Running mix_pair Unit Tests ===")
test_rng = np.random.default_rng(26052)
for i in range(200):
    c_dummy = test_rng.standard_normal(48000).astype(np.float32)
    n_dummy = test_rng.standard_normal(48000).astype(np.float32)
    target_snr = float(test_rng.uniform(-10.0, 30.0))
    
    c_out, n_out, mix_out = mix_pair(c_dummy, n_dummy, target_snr)
    
    # Assert peak safety
    assert np.max(np.abs(mix_out)) <= 0.99001, f"Peak exceeded: {np.max(np.abs(mix_out))}"
    
    # Assert SNR within +/- 0.1 dB
    meas_c_pow = np.mean(c_out ** 2)
    meas_n_pow = np.mean(n_out ** 2)
    meas_snr = 10.0 * np.log10(meas_c_pow / max(meas_n_pow, 1e-12))
    assert abs(meas_snr - target_snr) < 0.10, f"SNR mismatch: target={target_snr:.2f}, measured={meas_snr:.2f}"

# Zero-noise identity assertion
c_dummy = test_rng.standard_normal(48000).astype(np.float32)
c_z, n_z, mix_z = mix_pair(c_dummy, None, None)
assert np.allclose(c_z, mix_z), "Zero-noise failed clean == noisy identity!"
assert np.allclose(n_z, 0.0), "Zero-noise noise component is non-zero!"
print("mix_pair unit test PASSED (200 random pairs within +/- 0.1 dB SNR, zero-noise identity verified).\n")


In [ ]:
# === CELL 5: DATASET SETUP, FAST ZIP-STREAMING, EXCLUSIONS & SPLITS (Fix 2, Fix 5, Fix 6) ===
# Fix 6: Crop 2.0s window BEFORE resampling directly on BytesIO at 16 kHz rate.
# Fix 2: Remove silent zeros fallback in resolve_and_load_clip; partition via metadata.csv audio_class.
# Fix 5: LibriSpeech numeric speaker parsing, file stem matching, case-insensitive gun matching, and IoBT audit.

import csv, zipfile, io, random
from collections import defaultdict, Counter
import soundfile as sf
import scipy.signal as signal
from pathlib import Path

# ── 1. Isolated Zip Reading Helper ──
# Note: Using clean context-managed ZipFile reads per clip avoids stale FUSE file descriptor lockups across worker processes.

# ── 2. Fast Audio Reader (Crop BEFORE Resampling) (Fix 6) ──
def read_and_crop_audio(bio, target_sr=DF_SR, clip_len_samples=CLIP_LEN_SAMPLES):
    '''
    Reads audio from BytesIO, crops to target duration at native rate, then resamples.
    Drastically reduces resampling latency from entire clip to exactly the 2.0s segment.
    '''
    with sf.SoundFile(bio) as sfile:
        native_sr = sfile.samplerate
        total_frames = len(sfile)
        needed_frames = int(round((clip_len_samples / target_sr) * native_sr))
        
        if total_frames < needed_frames:
            # Short clip: read all, tile to required length
            data = sfile.read(dtype='float32')
            if data.ndim > 1:
                data = data.mean(axis=1)
            reps = math.ceil(needed_frames / max(1, total_frames))
            crop_native = np.tile(data, reps)[:needed_frames]
        else:
            # Long clip: pick random start frame and read ONLY needed_frames
            max_start = total_frames - needed_frames
            start_frame = random.randrange(max_start + 1)
            sfile.seek(start_frame)
            crop_native = sfile.read(frames=needed_frames, dtype='float32')
            if crop_native.ndim > 1:
                crop_native = crop_native.mean(axis=1)
            
            # Energy check: if random window landed in silent gap (RMS <= 1e-4), retry or center on peak
            if float(np.sqrt(np.mean(crop_native ** 2))) <= 1e-4:
                found_active = False
                for _ in range(3):
                    cand_start = random.randrange(max_start + 1)
                    sfile.seek(cand_start)
                    cand_crop = sfile.read(frames=needed_frames, dtype='float32')
                    if cand_crop.ndim > 1:
                        cand_crop = cand_crop.mean(axis=1)
                    if float(np.sqrt(np.mean(cand_crop ** 2))) > 1e-4:
                        crop_native = cand_crop
                        found_active = True
                        break
                if not found_active:
                    sfile.seek(0)
                    full_audio = sfile.read(dtype='float32')
                    if full_audio.ndim > 1:
                        full_audio = full_audio.mean(axis=1)
                    if len(full_audio) > 0 and float(np.sqrt(np.mean(full_audio ** 2))) > 1e-4:
                        peak_idx = int(np.argmax(np.abs(full_audio)))
                        start = max(0, min(peak_idx - needed_frames // 2, max_start))
                        crop_native = full_audio[start : start + needed_frames]
    # Resample only the 2.0s crop
    if native_sr != target_sr:
        crop_48k = signal.resample_poly(crop_native, target_sr, native_sr).astype(np.float32)
    else:
        crop_48k = crop_native.astype(np.float32)
        
    # Ensure exact length
    if len(crop_48k) < clip_len_samples:
        crop_48k = np.pad(crop_48k, (0, clip_len_samples - len(crop_48k)))
    else:
        crop_48k = crop_48k[:clip_len_samples]
        
    return crop_48k

# ── 3. Clip Resolver Without Silent Failures (Fix 2) ──
def resolve_and_load_clip(row, sih_root, clip_len_samples=CLIP_LEN_SAMPLES):
    '''
    Loads clip from zip archive or loose file.
    Fix 2: REMOVES silent zero-array fallback. Raises RuntimeError on any read error.
    '''
    arch = row.get('archive_name')
    ip = row.get('internal_path')
    
    # 1. Direct Zip Reading (Safe isolated handle per read to prevent FUSE lockups)
    if arch and (sih_root / arch).exists():
        try:
            with zipfile.ZipFile(str(sih_root / arch), 'r') as zf:
                with zf.open(ip) as zf_entry:
                    bio = io.BytesIO(zf_entry.read())
                    return read_and_crop_audio(bio, target_sr=DF_SR, clip_len_samples=clip_len_samples)
        except Exception as e:
            raise RuntimeError(f"Error reading zip entry {arch}/{ip}: {e}")

    # 2. Loose file fallback
    for cand in [sih_root / row.get('dataset_source', '') / ip, sih_root / row.get('filename', '')]:
        if cand.exists():
            try:
                with open(cand, 'rb') as fp:
                    bio = io.BytesIO(fp.read())
                    return read_and_crop_audio(bio, target_sr=DF_SR, clip_len_samples=clip_len_samples)
            except Exception as e:
                raise RuntimeError(f"Error reading loose audio file {cand}: {e}")
                
    raise FileNotFoundError(f"Clip not found in zip or filesystem: archive={arch}, internal_path={ip}, sih_root={sih_root}")

# ── 4. Load Metadata & Apply Exclusions (Fix 2, Fix 5) ──
assert SIH_DS is not None, "sih-26-processed-audio dataset not found in /kaggle/input"
with open(SIH_DS / 'metadata.csv', newline='', encoding='utf-8') as f:
    sih_rows = list(csv.DictReader(f))
print(f"Total raw metadata entries in sih-26: {len(sih_rows):,}")

# Load canonical exclusions (Task 1b: Mandatory loading)
canonical_clean_speakers = set()
canonical_clean_stems    = set()
canonical_clean_folders  = set()
canonical_gun_types      = set()
canonical_noise_stems    = set()
canonical_noise_folders  = set()

if CANONICAL_EXCLUSIONS_PATH and Path(CANONICAL_EXCLUSIONS_PATH).exists():
    with open(CANONICAL_EXCLUSIONS_PATH, newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            kind = r.get('kind', '').strip()
            val = r.get('value', '').strip()
            role = r.get('role', '').strip().lower()
            if role == 'clean':
                if kind == 'speaker':
                    canonical_clean_speakers.add(val)
                elif kind == 'file':
                    canonical_clean_stems.add(Path(val).stem.lower())
                elif kind == 'folder':
                    canonical_clean_folders.add(val.lower())
            else:  # noise
                if kind == 'speaker':
                    canonical_gun_types.add(val.lower())
                elif kind == 'file':
                    canonical_noise_stems.add(Path(val).stem.lower())
                elif kind == 'folder':
                    canonical_noise_folders.add(val.lower())
else:
    if not ALLOW_NO_EXCLUSIONS:
        raise FileNotFoundError("CRITICAL ERROR: canonical_exclusions.csv not found or unreadable! (ALLOW_NO_EXCLUSIONS=False)")
    print("WARNING: canonical_exclusions.csv not found; proceeding with empty exclusion sets.")

# Task 1b: Fail-safe stop if exclusions CSV has zero speaker rows
if len(canonical_clean_speakers) == 0 and not ALLOW_NO_EXCLUSIONS:
    raise ValueError(
        "CRITICAL ERROR: canonical_exclusions.csv loaded 0 clean speaker rows! "
        "Exclusions are invalid. Set ALLOW_NO_EXCLUSIONS = True if intentional."
    )

print(f"Canonical Exclusions Loaded (Tasks 1a & 1b):")
print(f"  - Clean unique speakers : {len(canonical_clean_speakers)}")
print(f"  - Clean unique stems    : {len(canonical_clean_stems)}")
print(f"  - Clean unique folders  : {len(canonical_clean_folders)}")
print(f"  - Gun types (noise)     : {len(canonical_gun_types)}")
print(f"  - Noise unique stems    : {len(canonical_noise_stems)}")
print(f"  - Noise unique folders  : {len(canonical_noise_folders)}")

# Task 1c: Robust LibriSpeech numeric speaker extractor from internal_path and filename
def extract_librispeech_speaker(internal_path, filename=''):
    for p_str in [internal_path, filename]:
        if not p_str:
            continue
        p = Path(p_str)
        parts = p.parts
        for i, part in enumerate(parts):
            if part.isdigit() and i + 1 < len(parts) and parts[i + 1].isdigit():
                return part
        if len(parts) >= 3 and parts[-3].isdigit():
            return parts[-3]
        if len(parts) >= 2 and parts[-2].isdigit():
            return parts[-2]
        stem = p.stem
        if '-' in stem:
            first = stem.split('-')[0]
            if first.isdigit():
                return first
    return None

# Task 1c: Audit LibriSpeech speech rows in SIH-26
librispeech_speech_rows = []
librispeech_speaker_ids = set()
for r in sih_rows:
    src = r.get('dataset_source', '').strip()
    cls = r.get('audio_class', '').strip().lower()
    ip = r.get('internal_path', '')
    fn = r.get('filename', '')
    if src == 'LibriSpeech' and cls == 'clean_speech':
        librispeech_speech_rows.append(r)
        spk = extract_librispeech_speaker(ip, fn)
        if spk:
            librispeech_speaker_ids.add(spk)

canonical_speakers_in_sih = canonical_clean_speakers.intersection(librispeech_speaker_ids)
print(f"\nSIH-26 LibriSpeech Audit (Task 1c):")
print(f"  - SIH-26 speech rows from LibriSpeech : {len(librispeech_speech_rows):,}")
print(f"  - Distinct LibriSpeech speaker IDs   : {len(librispeech_speaker_ids):,}")
print(f"  - Canonical speakers appearing in SIH: {len(canonical_speakers_in_sih)} / {len(canonical_clean_speakers)} ({sorted(list(canonical_speakers_in_sih))})")

# Task 1d: Check whether canonical clean file stems appear among SIH-26 filenames
sih_clean_stems = set(Path(r['filename']).stem.lower() for r in sih_rows if r.get('audio_class', '').strip().lower() == 'clean_speech')
clean_stem_overlap = canonical_clean_stems.intersection(sih_clean_stems)
print(f"\nCanonical Clean File Stem Audit (Task 1d):")
print(f"  - Canonical clean file stems in SIH-26 filenames: {len(clean_stem_overlap)}")

# Filter rows and record rule-by-rule drop counts (Task 1c)
sih_filtered = []
rule_speaker_drops = 0
rule_stem_drops    = 0
rule_folder_drops  = 0
rule_gun_drops     = 0
iobt_records_count = 0

for r in sih_rows:
    ip = r.get('internal_path', '')
    fn = r.get('filename', '')
    src = r.get('dataset_source', '')
    subclass = r.get('inferred_subclass', '')
    parent = r.get('parent_folder', '').lower()
    parts_lower = [part.lower() for part in Path(ip).parts]
    
    # Check for IoBT gunfire presence
    if 'iobt' in src.lower() or 'iobt' in ip.lower() or 'iobt' in subclass.lower():
        iobt_records_count += 1
        
    # 1. Speaker rule (clean speech)
    spk_id = extract_librispeech_speaker(ip, fn)
    if spk_id and spk_id in canonical_clean_speakers:
        rule_speaker_drops += 1
        continue
        
    # 2. File stem rule
    stem_ip = Path(ip).stem.lower()
    stem_fn = Path(fn).stem.lower()
    if (stem_ip in canonical_clean_stems or stem_fn in canonical_clean_stems or
        stem_ip in canonical_noise_stems or stem_fn in canonical_noise_stems):
        rule_stem_drops += 1
        continue
        
    # 3. Folder rule
    if (parent in canonical_clean_folders or any(p in canonical_clean_folders for p in parts_lower) or
        parent in canonical_noise_folders or any(p in canonical_noise_folders for p in parts_lower)):
        rule_folder_drops += 1
        continue
        
    # 4. Gun type rule (case-insensitive)
    is_gun_match = False
    for gun_name in canonical_gun_types:
        if gun_name in subclass.lower() or gun_name in ip.lower():
            is_gun_match = True
            break
    if is_gun_match:
        rule_gun_drops += 1
        continue
        
    sih_filtered.append(r)

final_speech_rows = [r for r in sih_filtered if r.get('audio_class', '').strip().lower() == 'clean_speech']
final_noise_rows  = [r for r in sih_filtered if r.get('audio_class', '').strip().lower() == 'noise']

print(f"\nCanonical Exclusions Rule-by-Rule Audit (Task 1c):")
print(f"  - Speaker exclusion rule removed  : {rule_speaker_drops:,} rows")
print(f"  - File stem exclusion rule removed: {rule_stem_drops:,} rows")
print(f"  - Folder exclusion rule removed   : {rule_folder_drops:,} rows")
print(f"  - Gun type exclusion rule removed : {rule_gun_drops:,} rows")
print(f"  - Total rows removed by rules     : {len(sih_rows) - len(sih_filtered):,} rows")
print(f"  - Final clean speech row count    : {len(final_speech_rows):,}")
print(f"  - Final noise row count           : {len(final_noise_rows):,}")
print(f"  - Total retained rows             : {len(sih_filtered):,}")

# Task 1c Assertion: 0 canonical speakers remain in retained pool
remaining_spks = set()
for r in sih_filtered:
    spk = extract_librispeech_speaker(r.get('internal_path', ''), r.get('filename', ''))
    if spk and spk in canonical_clean_speakers:
        remaining_spks.add(spk)
assert len(remaining_spks) == 0, f"ASSERTION FAILED: {len(remaining_spks)} canonical clean speakers still present after exclusion!"
print(f"ASSERTION PASSED: Count of canonical speakers still present after exclusion is strictly 0.\n")

# ── 5. Partition by audio_class Column & Assert Non-Empty (Fix 2) ──
print("=== Metadata Distribution (Fix 2) ===")
counts_audio_class = Counter(r.get('audio_class', '').strip().lower() for r in sih_filtered)
counts_dataset_source = Counter(r.get('dataset_source', '').strip() for r in sih_filtered)
counts_subclass = Counter(r.get('inferred_subclass', '').strip() for r in sih_filtered)

print("Counts per audio_class:")
for k, v in counts_audio_class.items():
    print(f"  - {k or 'MISSING'}: {v:,}")

print("\nCounts per dataset_source:")
for k, v in counts_dataset_source.most_common():
    print(f"  - {k or 'MISSING'}: {v:,}")

print("\nTop 20 inferred_subclass values:")
for k, v in counts_subclass.most_common(20):
    print(f"  - {k or 'MISSING'}: {v:,}")

# Partition speech vs noise based on audio_class column
speech_rows = [r for r in sih_filtered if r.get('audio_class', '').strip().lower() == 'clean_speech']
noise_all_rows = [r for r in sih_filtered if r.get('audio_class', '').strip().lower() == 'noise']

# Sub-partition defense noise domains
gunfire_rows = [r for r in noise_all_rows if 'firearms' in r.get('dataset_source', '').lower() or 'gun' in r.get('dataset_source', '').lower() or 'gun' in r.get('inferred_subclass', '').lower()]
drone_rows   = [r for r in noise_all_rows if 'drone' in r.get('dataset_source', '').lower() or 'drone' in r.get('inferred_subclass', '').lower()]
vehicle_rows = [r for r in noise_all_rows if 'vehicle' in r.get('dataset_source', '').lower() or 'vehicle' in r.get('inferred_subclass', '').lower()]
env_rows     = [r for r in noise_all_rows if 'demand' in r.get('dataset_source', '').lower() or 'env' in r.get('dataset_source', '').lower() or 'environmental' in r.get('inferred_subclass', '').lower()]

# If any noise rows fell outside the 4 bins, allocate them to env
categorized_noise_ids = {id(r) for r in (gunfire_rows + drone_rows + vehicle_rows + env_rows)}
remaining_noise = [r for r in noise_all_rows if id(r) not in categorized_noise_ids]
if remaining_noise:
    env_rows.extend(remaining_noise)

print(f"\nSubsets Partitioning Summary:")
print(f"  - Speech  : {len(speech_rows):,} rows")
print(f"  - Gunfire : {len(gunfire_rows):,} rows")
print(f"  - Drone   : {len(drone_rows):,} rows")
print(f"  - Vehicle : {len(vehicle_rows):,} rows")
print(f"  - Env     : {len(env_rows):,} rows")

# Strict assertion: every subset MUST be non-empty (Fix 2)
assert len(speech_rows) > 0, "Speech subset is empty!"
assert len(gunfire_rows) > 0, "Gunfire subset is empty!"
assert len(drone_rows) > 0, "Drone subset is empty!"
assert len(vehicle_rows) > 0, "Vehicle subset is empty!"
assert len(env_rows) > 0, "Environmental noise subset is empty!"
print("All subsets verified NON-EMPTY.\n")

# ── 6. Leak-Free Train / Validation Group Split (10% Held Out) ──
def group_split(rows, group_key_fn, val_ratio=0.10, seed=SEED):
    grouped = defaultdict(list)
    for r in rows:
        grouped[group_key_fn(r)].append(r)
    groups = sorted(grouped.keys())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_val_groups = max(1, int(round(len(groups) * val_ratio)))
    val_groups = set(groups[:n_val_groups])
    train_groups = set(groups[n_val_groups:])
    
    # Assert mutual exclusion
    assert train_groups.isdisjoint(val_groups), "Data leakage: train and val groups overlap!"
    
    train_rows = [r for g in train_groups for r in grouped[g]]
    val_rows   = [r for g in val_groups   for r in grouped[g]]
    return train_rows, val_rows, len(train_groups), len(val_groups)

speech_tr,  speech_va,  n_spk_tr, n_spk_va = group_split(speech_rows, lambda r: extract_librispeech_speaker(r.get('internal_path','')) or r.get('inferred_subclass',''))
gunfire_tr, gunfire_va, n_gf_tr,  n_gf_va  = group_split(gunfire_rows, lambda r: r.get('inferred_subclass','') or Path(r.get('internal_path','')).stem)
drone_tr,   drone_va,   n_dr_tr,  n_dr_va  = group_split(drone_rows,   lambda r: r.get('dataset_source','') + '__' + r.get('inferred_subclass',''))
vehicle_tr, vehicle_va, n_vh_tr,  n_vh_va  = group_split(vehicle_rows, lambda r: r.get('dataset_source','') + '__' + r.get('inferred_subclass',''))
env_tr,     env_va,     n_ev_tr,  n_ev_va  = group_split(env_rows,     lambda r: r.get('dataset_source','') + '__' + r.get('inferred_subclass',''))

print(f"Leak-Free Split Statistics (10% Groups Held Out):")
print(f"  - Speech  : {len(speech_tr):,} train ({n_spk_tr} groups) | {len(speech_va):,} val ({n_spk_va} groups)")
print(f"  - Gunfire : {len(gunfire_tr):,} train ({n_gf_tr} groups) | {len(gunfire_va):,} val ({n_gf_va} groups)")
print(f"  - Drone   : {len(drone_tr):,} train ({n_dr_tr} groups) | {len(drone_va):,} val ({n_dr_va} groups)")
print(f"  - Vehicle : {len(vehicle_tr):,} train ({n_vh_tr} groups) | {len(vehicle_va):,} val ({n_vh_va} groups)")
print(f"  - Env     : {len(env_tr):,} train ({n_ev_tr} groups) | {len(env_va):,} val ({n_ev_va} groups)\n")

# Task 1c Assertion: 0 canonical clean speakers in train or val pool
for pool_name, pool in [("training", speech_tr), ("validation", speech_va)]:
    spks_in_pool = set()
    for r in pool:
        spk = extract_librispeech_speaker(r.get("internal_path", ""), r.get("filename", ""))
        if spk and spk in canonical_clean_speakers:
            spks_in_pool.add(spk)
    assert len(spks_in_pool) == 0, f"ASSERTION FAILED: {len(spks_in_pool)} canonical clean speakers found in {pool_name} pool!"
print(f"ASSERTION PASSED: Canonical clean speakers count in training and validation pools is strictly 0.\n")


In [ ]:
# === CELL 6: STARTUP SMOKE CELL (Fix 2) ===
# Loads 20 random rows per subset (speech, gunfire, drone, vehicle, env).
# Prints archive_name/internal_path, asserts length == 96,000, RMS > 1e-4, and no NaN/Inf.
# Aborts with a clear message on any failure.

print("=== Running Startup Smoke Test (20 Random Rows per Subset) ===")
silent_warning_count = 0
subsets = {
    'speech': speech_tr,
    'gunfire': gunfire_tr,
    'drone': drone_tr,
    'vehicle': vehicle_tr,
    'env': env_tr
}

for subset_name, row_list in subsets.items():
    print(f"\n--- Verifying subset: {subset_name.upper()} ({len(row_list)} available) ---")
    sample_rows = random.sample(row_list, min(20, len(row_list)))
    for idx, row in enumerate(sample_rows, 1):
        arch = row.get('archive_name')
        ip = row.get('internal_path')
        print(f"  [{idx:02d}/20] Loading: {arch} :: {ip}")
        
        try:
            clip = resolve_and_load_clip(row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
        except Exception as e:
            raise RuntimeError(f"STARTUP SMOKE FAILURE on subset '{subset_name}', row {arch}::{ip}: {e}")
            
        # Assert exact length
        if len(clip) != CLIP_LEN_SAMPLES:
            raise AssertionError(f"Length mismatch on {ip}: expected {CLIP_LEN_SAMPLES}, got {len(clip)}")
            
        # Assert finite values
        if np.isnan(clip).any() or np.isinf(clip).any():
            raise AssertionError(f"NaN or Inf encountered in audio clip: {arch}::{ip}")
            
        # Assert audible signal RMS > 1e-4
        rms = float(np.sqrt(np.mean(clip ** 2)))
        if rms <= 1e-4:
            silent_warning_count += 1
            print(f"  [WARNING] Low-energy/silent clip detected (RMS={rms:.2e} <= 1e-4): {arch}::{ip}")

if silent_warning_count > 5:
    raise AssertionError(f"Too many silent clips detected in startup test ({silent_warning_count}/100)! Check dataset integrity.")

print()
print(f"Startup Smoke Test PASSED: All 100 sample clips (20/subset) loaded, verified finite, length={CLIP_LEN_SAMPLES}, silent_warnings={silent_warning_count}/100.")
print()


In [ ]:
# === CELL 7: EMPIRICAL DIAGNOSTIC ON REAL CLIPS (Fix 7) ===
# Removes hardcoded conclusion text.
# Evaluates 1,000 pairs of REAL clips from the attached dataset (speech and noise).
# Compares OLD verbatim mixing logic vs NEW mix_pair, printing the exact measured distributions.

def mix_pair_old_verbatim(c, n, snr_db):
    '''OLD verbatim mixing logic from deepfilternet3_v2_sih_defence_augmented.ipynb'''
    c = c / (np.sqrt(np.mean(c**2)) + 1e-8)
    n = n / (np.sqrt(np.mean(n**2)) + 1e-8)
    n = n * (10 ** (-snr_db / 20))
    noisy = c + n
    peak = float(np.max(np.abs(noisy)))
    noisy = noisy / max(1.0, peak)
    clipped = peak > 1.0
    return c, noisy, clipped

print("=== Running Pre-flight Diagnostic on 1,000 REAL Audio Clip Pairs ===")
snr_grid = [-10, -5, 0, 5, 10, 15, 20, 25]
num_pairs_per_snr = 125 # 125 * 8 = 1,000 pairs total

all_noise_sample = gunfire_tr + drone_tr + vehicle_tr + env_tr
t0 = time.time()

print("\n| SNR (dB) | Old Peak > 1 % | OLD 20*log10(RMS_t / RMS_ny) [p5, Med, p95] | NEW 20*log10(RMS_t / RMS_ny) [p5, Med, p95] |")
print("| :---: | :---: | :---: | :---: |")

for snr in snr_grid:
    old_peaks = []
    old_ratios = []
    new_ratios = []
    
    for _ in range(num_pairs_per_snr):
        c_row = random.choice(speech_tr)
        n_row = random.choice(all_noise_sample)
        
        c_audio = resolve_and_load_clip(c_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
        n_audio = resolve_and_load_clip(n_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
        
        c_old, ny_old, clipped = mix_pair_old_verbatim(c_audio.copy(), n_audio.copy(), snr)
        c_new, _, ny_new = mix_pair(c_audio.copy(), n_audio.copy(), snr)
        
        old_peaks.append(clipped)
        old_ratios.append(20.0 * np.log10((np.sqrt(np.mean(c_old**2)) + 1e-12) / (np.sqrt(np.mean(ny_old**2)) + 1e-12)))
        new_ratios.append(20.0 * np.log10((np.sqrt(np.mean(c_new**2)) + 1e-12) / (np.sqrt(np.mean(ny_new**2)) + 1e-12)))
        
    old_clip_pct = np.mean(old_peaks) * 100.0
    mean_o, med_o = np.mean(old_ratios), np.median(old_ratios)
    p5_o, p95_o = np.percentile(old_ratios, 5), np.percentile(old_ratios, 95)
    
    mean_n, med_n = np.mean(new_ratios), np.median(new_ratios)
    p5_n, p95_n = np.percentile(new_ratios, 5), np.percentile(new_ratios, 95)
    
    print(f"| {snr:+3d} dB | {old_clip_pct:5.1f}% | Mean {mean_o:+5.2f} dB [{p5_o:+5.2f}, {med_o:+5.2f}, {p95_o:+5.2f}] | Mean {mean_n:+5.2f} dB [{p5_n:+5.2f}, {med_n:+5.2f}, {p95_n:+5.2f}] |")

print(f"\nDiagnostic completed in {time.time() - t0:.2f}s.\n")


In [ ]:
# === CELL 8: FIXED STRATIFIED VALIDATION MIXTURE SET (Fix 3) ===
# Precomputes exactly 100 validation mixtures ONCE with a seeded RNG.
# Stratified: >= 8 clips per SNR in [-10, -5, 0, 5, 10, 15, 20, 25],
# 10 near-clean, 5 zero-noise, and ~40% gunfire noise.
# Retained resident in memory as arrays. Evaluates pretrained baseline once.

print("=== Precomputing Fixed 100-Clip Validation Set (Fix 3) ===")
val_rng = random.Random(26052)
val_np_rng = np.random.default_rng(26052)

val_fixed_clips = []
# 1. 85 Standard Grid clips: distributed across 8 SNRs (>= 10 clips per SNR >= 8)
snr_tiers = [-10, -5, 0, 5, 10, 15, 20, 25]
counts_per_tier = [11, 11, 11, 11, 11, 10, 10, 10] # Sums to 85

# Total noisy clips = 85 + 10 (near clean) = 95.
# Exactly 38 gunfire clips (~40% of 95), remaining 57 split evenly across drone (19), vehicle (19), env (19).
noisy_noise_classes = (['gunfire'] * 38) + (['drone'] * 19) + (['vehicle'] * 19) + (['env'] * 19)
val_rng.shuffle(noisy_noise_classes)

noise_idx = 0

for tier_idx, snr_val in enumerate(snr_tiers):
    n_in_tier = counts_per_tier[tier_idx]
    for _ in range(n_in_tier):
        n_class = noisy_noise_classes[noise_idx]
        noise_idx += 1
        
        c_row = val_rng.choice(speech_va)
        if n_class == 'gunfire': n_row = val_rng.choice(gunfire_va)
        elif n_class == 'drone': n_row = val_rng.choice(drone_va)
        elif n_class == 'vehicle': n_row = val_rng.choice(vehicle_va)
        else: n_row = val_rng.choice(env_va)
        
        c_audio = resolve_and_load_clip(c_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
        n_audio = resolve_and_load_clip(n_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
        c_mix, n_mix, ny_mix = mix_pair(c_audio, n_audio, snr_val)
        val_fixed_clips.append({
            'clean': c_mix,
            'noisy': ny_mix,
            'snr': float(snr_val),
            'type': n_class,
            'stratum': f"{int(snr_val):+d}dB",
            'is_gunfire': (n_class == 'gunfire')
        })

# 2. 10 Near-Clean clips (SNR 30 - 40 dB)
for _ in range(10):
    n_class = noisy_noise_classes[noise_idx]
    noise_idx += 1
    c_row = val_rng.choice(speech_va)
    if n_class == 'gunfire': n_row = val_rng.choice(gunfire_va)
    elif n_class == 'drone': n_row = val_rng.choice(drone_va)
    elif n_class == 'vehicle': n_row = val_rng.choice(vehicle_va)
    else: n_row = val_rng.choice(env_va)
    
    c_audio = resolve_and_load_clip(c_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
    n_audio = resolve_and_load_clip(n_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
    near_clean_snr = float(val_rng.uniform(30.0, 40.0))
    c_mix, n_mix, ny_mix = mix_pair(c_audio, n_audio, near_clean_snr)
    val_fixed_clips.append({
        'clean': c_mix,
        'noisy': ny_mix,
        'snr': near_clean_snr,
        'type': f"near_clean_{n_class}",
        'stratum': 'near_clean',
        'is_gunfire': (n_class == 'gunfire')
    })

# 3. 5 Zero-Noise clips (clean == noisy)
for _ in range(5):
    c_row = val_rng.choice(speech_va)
    c_audio = resolve_and_load_clip(c_row, SIH_DS, clip_len_samples=CLIP_LEN_SAMPLES)
    c_mix, n_mix, ny_mix = mix_pair(c_audio, None, None)
    val_fixed_clips.append({
        'clean': c_mix,
        'noisy': ny_mix,
        'snr': 99.0,
        'type': 'zero_noise',
        'stratum': 'zero_noise',
        'is_gunfire': False
    })

assert len(val_fixed_clips) == 100, f"Expected 100 validation clips, got {len(val_fixed_clips)}"
print(f"Fixed validation set constructed: exactly {len(val_fixed_clips)} clips in memory.")
print(f"  - Standard grid clips: 85 (>= 10 per tier across {snr_tiers})")
print(f"  - Near-clean clips   : 10 (30-40 dB SNR)")
print(f"  - Zero-noise clips   : 5")
print(f"  - Gunfire noise count: {sum(1 for x in val_fixed_clips if 'gunfire' in x['type'])} / 95 noisy clips (40.0%)\n")

# Evaluation metric helpers
def calc_metrics_for_eval(clean_48k, est_48k):
    # Downsample to 16 kHz for standard metric alignment
    c_16k = signal.resample_poly(clean_48k, 16000, DF_SR).astype(np.float32)
    e_16k = signal.resample_poly(est_48k, 16000, DF_SR).astype(np.float32)
    min_len = min(len(c_16k), len(e_16k))
    c, e = c_16k[:min_len], e_16k[:min_len]
    
    # SNR
    snr = 10.0 * np.log10((np.sum(c ** 2) + 1e-12) / (np.sum((c - e) ** 2) + 1e-12))
    
    # SI-SDR
    a = np.dot(c, e) / (np.dot(c, c) + 1e-12)
    e_target = a * c
    sisdr = 10.0 * np.log10((np.sum(e_target ** 2) + 1e-12) / (np.sum((e - e_target) ** 2) + 1e-12))
    
    # STOI
    try:
        st = float(pystoi_stoi(c, e, 16000, extended=False))
    except Exception:
        st = float('nan')
        
    # PESQ
    try:
        pq = float(pesq_wb(16000, c, e, 'wb'))
    except Exception:
        pq = float('nan')
        
    # Active Level Ratio
    frame_len = int(16000 * 0.03) # 30ms frames
    n_frames = min_len // frame_len
    if n_frames > 0:
        c_f = c[:n_frames * frame_len].reshape(n_frames, frame_len)
        e_f = e[:n_frames * frame_len].reshape(n_frames, frame_len)
        rms_c = np.sqrt(np.mean(c_f ** 2, axis=1) + 1e-12)
        thresh = max(np.max(rms_c) * (10.0 ** (-30.0 / 20.0)), 10.0 ** (-45.0 / 20.0))
        act = rms_c > thresh
        if np.any(act):
            r_c = np.sqrt(np.mean(c_f[act] ** 2) + 1e-12)
            r_e = np.sqrt(np.mean(e_f[act] ** 2) + 1e-12)
            lvl_ratio = float(20.0 * np.log10(r_e / r_c))
        else:
            lvl_ratio = 0.0
    else:
        lvl_ratio = 0.0
        
    return snr, sisdr, st, pq, lvl_ratio


In [ ]:
# === CELL 9: AUDIOADAPTER, MULTI-GPU & SI-SDR LOSS TEST (Fix 4, Fix 8) ===
# Fix 8: USE_DP flag with one-batch forward/backward pass verification.
# Fix 4: Differentiable time-domain SI-SDR loss with DeepFilterNet's Vorbis analysis window.
# Fix 4: Unit test asserting SI-SDR > 30 dB on 50 clean clips.

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 1. Load Initial Model
print("=== Initializing DeepFilterNet3 Model ===")
base_model_dir = maybe_download_model("DeepFilterNet3")
model, df_state, _ = init_df(base_model_dir, log_file=None)
model = model.to(device)

init_target = RESUME_PATH if (RESUME_PATH and Path(RESUME_PATH).exists()) else (MODEL_INIT if MODEL_INIT != "pretrained" and Path(MODEL_INIT).exists() else None)
if init_target is not None:
    print(f"Loading weights from {init_target}...")
    if str(init_target).endswith('.hk'):
        init_state = hk.load_file(str(init_target))
        init_state = {k: torch.from_numpy(v) if not isinstance(v, torch.Tensor) else v for k, v in init_state.items()}
    else:
        init_state = torch.load(str(init_target), map_location='cpu', weights_only=False)
    bare_init_model = model.module if isinstance(model, nn.DataParallel) else model
    bare_init_model.load_state_dict(init_state, strict=True)
    print(f"Weights loaded successfully from {init_target} (133 tensors restored).")
else:
    print("Initialized directly from clean PRETRAINED reference weights (MODEL_INIT='pretrained').")

# Extract Vorbis analysis window from df_state (Fix 4)
vorbis_win = torch.from_numpy(df_state.fft_window()).to(device)
print(f"DeepFilterNet STFT Analysis Window: length={len(vorbis_win)}, type=Vorbis")

def calc_time_sisdr_loss(pred_spec_comp, target_audio, win=vorbis_win, n_fft=960, hop=480, eps=1e-8):
    '''
    Differentiable time-domain SI-SDR loss computed via PyTorch inverse STFT.
    Input pred_spec_comp: [B, 1, T_frames, F, 2]
    Input target_audio  : [B, 1, T_samples]
    Uses DeepFilterNet's exact Vorbis window and hop size (480).
    '''
    # Complex tensor: [B, T_frames, F]
    comp = torch.view_as_complex(pred_spec_comp.squeeze(1))
    # Transpose for istft: [B, F, T_frames]
    comp_t = comp.transpose(1, 2)
    
    # Synthesize time-domain signal
    pred_time = torch.istft(
        comp_t,
        n_fft=n_fft,
        hop_length=hop,
        win_length=n_fft,
        window=win.to(pred_spec_comp.device),
        center=True
    )
    
    target_time = target_audio.squeeze(1)
    min_len = min(pred_time.shape[-1], target_time.shape[-1])
    p = pred_time[..., :min_len]
    t = target_time[..., :min_len]
    
    # Scale-Invariant SDR
    Rss = torch.sum(t ** 2, dim=-1, keepdim=True)
    a = torch.sum(t * p, dim=-1, keepdim=True) / (Rss + eps)
    e_true = a * t
    e_res = p - e_true
    
    norm_target = torch.sum(e_true ** 2, dim=-1)
    norm_res = torch.sum(e_res ** 2, dim=-1)
    sisdr = 10.0 * torch.log10((norm_target + eps) / (norm_res + eps))
    return -torch.mean(sisdr), torch.mean(sisdr)

def batch_df_features(audio_batch, df_state, nb_df):
    specs, erbs, specfs = [], [], []
    for i in range(audio_batch.shape[0]):
        item = audio_batch[i]
        if item.ndim == 1: item = item.unsqueeze(0)
        # libDF maintains running normalization statistics; isolate each clip.
        df_state.reset()
        s, e, sf = df_features(item.cpu(), df_state, nb_df)
        specs.append(s.squeeze(0))
        erbs.append(e.squeeze(0))
        specfs.append(sf.squeeze(0))
    return torch.stack(specs, dim=0), torch.stack(erbs, dim=0), torch.stack(specfs, dim=0)

class AudioAdapter(nn.Module):
    def __init__(self, net, lambda_sisdr=LAMBDA_SISDR):
        super().__init__()
        self.net = net
        self.lambda_sisdr = lambda_sisdr

    def forward(self, noisy_t, clean_t, df_state, device):
        net_bare = self.net.module if isinstance(self.net, nn.DataParallel) else self.net
        nb_df = net_bare.nb_df
        
        spec, erb, specf = batch_df_features(noisy_t, df_state, nb_df)
        target, _, _     = batch_df_features(clean_t, df_state, nb_df)
        
        spec   = spec.to(device)
        erb    = erb.to(device)
        specf  = specf.to(device)
        target = target.to(device)
        clean_audio = clean_t.to(device)
        
        pred = self.net(spec.clone(), erb, specf)[0]
        
        # 1. Complex STFT L1 Loss
        loss_complex = F.l1_loss(pred.float(), target.float())
        
        # 2. Spectral Magnitude Norm L1 Loss
        loss_mag = F.l1_loss(
            torch.linalg.vector_norm(pred.float(), dim=-1),
            torch.linalg.vector_norm(target.float(), dim=-1)
        )
        
        # 3. Differentiable Time-Domain SI-SDR Loss (Fix 4: raises without silent swallowing)
        if self.lambda_sisdr > 0.0:
            loss_sisdr_neg, sdr_val = calc_time_sisdr_loss(pred.float(), clean_audio, win=vorbis_win)
            loss_sisdr = loss_sisdr_neg
        else:
            loss_sisdr = torch.tensor(0.0, device=device)
            
        total_loss = loss_complex + 0.25 * loss_mag + self.lambda_sisdr * loss_sisdr
        return total_loss, loss_complex, loss_mag, loss_sisdr

# ── Multi-GPU Setup & DataParallel Verification (Fix 8) ──
if USE_DP and torch.cuda.device_count() >= 2:
    print(f"DataParallel requested across {torch.cuda.device_count()} GPUs. Testing forward/backward pass...")
    dp_candidate = nn.DataParallel(model).to(device)
    test_adapter = AudioAdapter(dp_candidate, lambda_sisdr=0.1).to(device)
    dummy_noisy = torch.randn(2, 144000)
    dummy_clean = torch.randn(2, 144000)
    try:
        t_loss, _, _, _ = test_adapter(dummy_noisy, dummy_clean, df_state, device)
        t_loss.backward()
        model = dp_candidate
        print(f"DataParallel verification PASSED: active across {torch.cuda.device_count()} GPUs.")
    except Exception as e:
        print(f"WARNING: DataParallel test failed ({e}); falling back to single GPU.")
        model = model.module if isinstance(model, nn.DataParallel) else model
        model = model.to(device)
        USE_DP = False
else:
    model = model.to(device)
    print(f"Running on single device: {device}")

adapter = AudioAdapter(model, lambda_sisdr=LAMBDA_SISDR).to(device)

# ── SI-SDR Loss Verification Test Cell (Fix 4) ──
print("\n=== Running SI-SDR Loss Verification Test (50 Clean Clips) ===")
net_bare = model.module if isinstance(model, nn.DataParallel) else model
sisdr_test_vals = []
for i in range(50):
    audio_test = torch.randn(1, 1, 144000)
    df_state.reset()
    spec_test, _, _ = df_features(audio_test.squeeze(0), df_state, net_bare.nb_df)
    _, sdr_val = calc_time_sisdr_loss(spec_test, audio_test, win=vorbis_win)
    sisdr_test_vals.append(sdr_val.item())

mean_test_sisdr = float(np.mean(sisdr_test_vals))
print(f"Tested 50 clean clips through STFT -> calc_time_sisdr_loss: Mean SI-SDR = {mean_test_sisdr:.2f} dB (Min = {min(sisdr_test_vals):.2f} dB)")
if mean_test_sisdr < 30.0:
    print("WARNING: SI-SDR reconstruction below 30 dB! Setting lambda_sisdr = 0.0.")
    adapter.lambda_sisdr = 0.0
else:
    print("SI-SDR loss verification PASSED (> 30 dB achieved cleanly).\n")

# ── Precompute Pretrained Baseline on Fixed 100 Validation Clips (Fix 3 & Task 3) ──
print("Pre-computing Pretrained Baseline on the 100 fixed validation clips...")
pre_scores = []
bare_eval_model = model.module if isinstance(model, nn.DataParallel) else model
bare_eval_model.eval()

with torch.no_grad():
    for item in val_fixed_clips:
        c_np = item['clean']
        ny_np = item['noisy']
        df_state.reset()
        enh_pre = enhance(bare_eval_model, df_state, torch.from_numpy(ny_np).unsqueeze(0)).squeeze().cpu().numpy()
        sc = calc_metrics_for_eval(c_np, enh_pre)
        pre_scores.append(sc)
        item['pre_sisdr'] = sc[1]

pre_mean = np.nanmean(pre_scores, axis=0)
strata_names = ['-10dB', '-5dB', '+0dB', '+5dB', '+10dB', '+15dB', '+20dB', '+25dB', 'near_clean', 'zero_noise']
pre_strata_sisdr = {}
for s_name in strata_names:
    s_vals = [item['pre_sisdr'] for item in val_fixed_clips if item['stratum'] == s_name]
    pre_strata_sisdr[s_name] = float(np.nanmean(s_vals))

pre_gun_sisdr = float(np.nanmean([item['pre_sisdr'] for item in val_fixed_clips if item['is_gunfire']]))

print(f"Pretrained Baseline Overall: SNR={pre_mean[0]:.2f}dB | SI-SDR={pre_mean[1]:.2f}dB | STOI={pre_mean[2]:.3f} | PESQ={pre_mean[3]:.2f} | LvlRatio={pre_mean[4]:+.2f}dB")
print(f"Pretrained Gunfire-Only SI-SDR (38 clips): {pre_gun_sisdr:.2f} dB")
print("Pretrained Per-Stratum SI-SDR Breakdown:")
for s_name in strata_names:
    print(f"  [{s_name:>10s}]: {pre_strata_sisdr[s_name]:.2f} dB")
print()


In [ ]:
# === CELL 10: 50-STEP TIMING BENCHMARK (Fix 6) ===
# Uses 4 workers, persistent_workers=True, and worker_init_fn.
# Measures steps per second and estimated hours for NUM_STEPS (10,000 steps).
# Warns if estimated runtime exceeds 10 hours.

def worker_init_fn(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

# Dynamic Mix Dataset
class V3_4_ZipMixSet(torch.utils.data.Dataset):
    def __init__(self, speech, gunfire, drone, vehicle, env, sih_root, count=10000, clip_len_samples=CLIP_LEN_SAMPLES):
        self.speech = speech
        self.gunfire = gunfire
        self.drone = drone
        self.vehicle = vehicle
        self.env = env
        self.sih_root = sih_root
        self.count = count
        self.clip_len = clip_len_samples
        self.read_fail_count = 0
        self.read_total_count = 0

    def __len__(self):
        return self.count

    def __getitem__(self, idx):
        self.read_total_count += 2
        try:
            c_row = self.speech[idx % len(self.speech)]
            c = resolve_and_load_clip(c_row, self.sih_root, self.clip_len)
        except Exception as e:
            self.read_fail_count += 1
            raise e
            
        r_val = random.random()
        # V3.4: Gunfire-biased mix - 60% gunfire (was 40%) to specialize on burst suppression
        if r_val < 0.60 and self.gunfire:
            n_row = random.choice(self.gunfire)
        elif r_val < 0.73 and self.drone:
            n_row = random.choice(self.drone)
        elif r_val < 0.87 and self.vehicle:
            n_row = random.choice(self.vehicle)
        else:
            n_row = random.choice(self.env)
            
        try:
            n = resolve_and_load_clip(n_row, self.sih_root, self.clip_len)
        except Exception as e:
            self.read_fail_count += 1
            raise e
            
        roll = random.random()
        if roll < 0.05:
            c_mix, n_mix, noisy = mix_pair(c, None, None)
        elif roll < 0.15:
            snr = random.uniform(30.0, 40.0)
            c_mix, n_mix, noisy = mix_pair(c, n, snr)
        else:
            snr = random.choice([-10.0, -5.0, 0.0, 5.0, 10.0, 15.0, 20.0, 25.0])
            c_mix, n_mix, noisy = mix_pair(c, n, snr)
            
        return {
            'clean': torch.from_numpy(c_mix).unsqueeze(0),
            'noisy': torch.from_numpy(noisy).unsqueeze(0)
        }

import copy

print("=== Running 50-Step Pre-Flight Timing Benchmark (Task 2) ===")
# Task 2 Fix: Deep-copy model state_dict (CPU) and optimizer state before benchmark
bare_model = model.module if isinstance(model, nn.DataParallel) else model
saved_model_state = {k: v.cpu().clone() for k, v in bare_model.state_dict().items()}

timing_ds = V3_4_ZipMixSet(speech_tr, gunfire_tr, drone_tr, vehicle_tr, env_tr, SIH_DS, count=100, clip_len_samples=CLIP_LEN_SAMPLES)
timing_loader = torch.utils.data.DataLoader(
    timing_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    worker_init_fn=worker_init_fn,
    drop_last=True
)

timing_opt = torch.optim.AdamW(adapter.parameters(), lr=1e-5)
saved_opt_state = copy.deepcopy(timing_opt.state_dict())

adapter.train()
t_iter = iter(timing_loader)

# Warmup 5 steps
for _ in range(5):
    try:
        b = next(t_iter)
    except StopIteration:
        t_iter = iter(timing_loader)
        b = next(t_iter)
    timing_opt.zero_grad(set_to_none=True)
    l, _, _, _ = adapter(b['noisy'], b['clean'], df_state, device)
    l.backward()
    timing_opt.step()

# Benchmark 50 steps
t_bench_start = time.time()
BENCH_STEPS = 50
for s in range(BENCH_STEPS):
    try:
        b = next(t_iter)
    except StopIteration:
        t_iter = iter(timing_loader)
        b = next(t_iter)
    timing_opt.zero_grad(set_to_none=True)
    l, _, _, _ = adapter(b['noisy'], b['clean'], df_state, device)
    l.backward()
    timing_opt.step()

t_bench_elapsed = time.time() - t_bench_start
steps_per_sec = BENCH_STEPS / max(t_bench_elapsed, 1e-6)
est_total_hours = (NUM_STEPS / steps_per_sec) / 3600.0

print(f"Timing Results: {BENCH_STEPS} steps completed in {t_bench_elapsed:.2f}s ({steps_per_sec:.2f} steps/s).")
print(f"Estimated total runtime for {NUM_STEPS} steps: {est_total_hours:.2f} hours.")
if est_total_hours > 10.0:
    print(f"WARNING: Estimated runtime ({est_total_hours:.2f}h) exceeds the 10-hour threshold!")
else:
    print(f"Pacing OK: Estimated runtime ({est_total_hours:.2f}h) is within the 10-hour limit.")

# Task 2 Fix: Restore model state_dict and optimizer state, reset step counter, assert parameter equality
bare_model.load_state_dict(saved_model_state)
timing_opt.load_state_dict(saved_opt_state)

# Assert with torch.equal that every parameter strictly equals the saved copy
current_state = bare_model.state_dict()
for p_name, saved_t in saved_model_state.items():
    cur_t = current_state[p_name].cpu()
    assert torch.equal(cur_t, saved_t), f"ASSERTION FAILED: Parameter '{p_name}' was altered during benchmark and not restored!"
print("Timing Benchmark State Isolation PASSED: All model parameters strictly match pre-benchmark weights via torch.equal assertion.\n")


In [ ]:
# === CELL 11: MAIN TRAINING LOOP WITH ALL CONTROLS (Fix 1, Fix 2, Fix 3, Fix 8, Fix 9) ===
# Fix 8: SMOKE mode (stops at step 30 + 1 eval pass).
# Fix 2: Shared counter stopping if load failures exceed 0.5%.
# Fix 9: Immediate abort on NaN/Inf loss; separate loss term logging; config.json.
# Fix 3: Fixed 100 validation mixtures scored side-by-side against pre-computed baseline.
# Fix 1: Dual checkpoint export (.hk verbatim + .pt state_dict).

def get_current_lr(step, total_steps=NUM_STEPS, warmup=WARMUP_STEPS, lr_max=LR_MAX, lr_min=LR_MIN):
    if step < warmup:
        return lr_max * float(step + 1) / float(max(1, warmup))
    progress = float(step - warmup) / float(max(1, total_steps - warmup))
    return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + math.cos(math.pi * progress))

# Training DataLoader
train_ds = V3_4_ZipMixSet(
    speech_tr, gunfire_tr, drone_tr, vehicle_tr, env_tr, SIH_DS,
    count=NUM_STEPS * BATCH_SIZE * 2,
    clip_len_samples=CLIP_LEN_SAMPLES
)

train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    worker_init_fn=worker_init_fn,
    drop_last=True
)

optimizer = torch.optim.AdamW(adapter.parameters(), lr=LR_MIN, weight_decay=1e-4)

# Resume support (Task 5c: step restored from RESUME_STEP, optimizer state NOT restored)
start_step = int(RESUME_STEP)
best_val_score = float(RESUME_BEST_SCORE)
if RESUME_PATH and Path(RESUME_PATH).exists():
    print(f"Resume requested from: {RESUME_PATH}")
    print(f"Restoring step counter to RESUME_STEP = {start_step}")
    print(f"Setting initial best_val_score to RESUME_BEST_SCORE = {best_val_score}")
    bare_net = model.module if isinstance(model, nn.DataParallel) else model
    if str(RESUME_PATH).endswith('.hk'):
        res_state = hk.load_file(str(RESUME_PATH))
        res_state = {k: torch.from_numpy(v) if not isinstance(v, torch.Tensor) else v for k, v in res_state.items()}
    else:
        res_state = torch.load(str(RESUME_PATH), map_location='cpu', weights_only=False)
    bare_net.load_state_dict(res_state, strict=True)
    print(f"Checkpoint weights restored into bare model from {RESUME_PATH} (optimizer state not restored; LR begins at step {start_step}).")

# Initialize validation log CSV (Task 3: includes strata and gunfire SI-SDR columns)
val_log_file = OUT_DIR / 'validation_log.csv'
val_log_header = [
    'step', 'lr', 'loss', 'loss_complex', 'loss_mag', 'loss_sisdr',
    'snr', 'sisdr', 'stoi', 'pesq', 'lvl_ratio',
    'pre_snr', 'pre_sisdr', 'pre_stoi', 'pre_pesq', 'pre_lvl_ratio',
    'gunfire_sisdr', 'pre_gunfire_sisdr'
] + [f'sisdr_{s}' for s in strata_names]

if not val_log_file.exists() or start_step == 0:
    with open(val_log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(val_log_header)

print(f"=== Starting Optimization for {NUM_STEPS} Steps (SMOKE={SMOKE}) ===")
target_steps = 30 if SMOKE else NUM_STEPS

step = start_step
saved_rolling_ckpts = []
train_iter = iter(train_loader)
t_start = time.time()
running_loss = 0.0
running_comp = 0.0
running_mag = 0.0
running_sdr = 0.0
step_times = []

adapter.train()

try:
    while step < target_steps:
        t_s0 = time.time()
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
        
        lr = get_current_lr(step)
        for g in optimizer.param_groups:
            g['lr'] = lr
        
        noisy_t = batch['noisy']
        clean_t = batch['clean']
    
        optimizer.zero_grad(set_to_none=True)
        loss, l_comp, l_mag, l_sdr = adapter(noisy_t, clean_t, df_state, device)
    
        # Fix 9: Stop immediately on NaN or Inf loss
        loss_val = float(loss.detach())
        if math.isnan(loss_val) or math.isinf(loss_val):
            raise ValueError(f"CRITICAL: Non-finite loss detected at step {step}: loss={loss_val}. Aborting immediately!")
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), GRAD_CLIP)
        optimizer.step()
    
        running_loss += loss_val
        running_comp += float(l_comp.detach())
        running_mag  += float(l_mag.detach())
        running_sdr  += float(l_sdr.detach())
    
        step += 1
        step_times.append(time.time() - t_s0)
        
        # Memory leak prevention: explicitly delete batch and intermediate loss tensors
        del batch, noisy_t, clean_t, loss, l_comp, l_mag, l_sdr
    
        # Fix 2: Check dataset read failure rate (< 0.5%)
        if train_ds.read_total_count > 100:
            fail_rate = train_ds.read_fail_count / max(1, train_ds.read_total_count)
            if fail_rate > 0.005:
                raise RuntimeError(f"ABORT: Audio load failure rate ({fail_rate*100:.2f}%) exceeded 0.5% tolerance threshold!")
            
        # Fix 9: Separate loss term logging and periodic garbage collection every 50 steps
        if step % 50 == 0 or (SMOKE and step == target_steps):
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            n_log = 50 if step % 50 == 0 else (step % 50)
            elapsed = time.time() - t_start
            print(f"[step {step:5d}/{target_steps}] (optimizer steps: {step}) lr={lr:.2e}  loss={running_loss/n_log:.4f} "
                  f"(comp={running_comp/n_log:.4f}, mag={running_mag/n_log:.4f}, sdr={running_sdr/n_log:.2f})  elapsed={elapsed:.0f}s")
            running_loss = 0.0; running_comp = 0.0; running_mag = 0.0; running_sdr = 0.0

        # Task 5b: Print GPU memory and process RAM every 100 steps
        if step % 100 == 0:
            ram_gb = psutil.Process().memory_info().rss / (1024**3)
            gpu_stats = []
            if torch.cuda.is_available():
                for d_idx in range(torch.cuda.device_count()):
                    mem_mb = torch.cuda.memory_allocated(d_idx) / (1024**2)
                    gpu_stats.append(f"GPU{d_idx}:{mem_mb:.1f}MB")
            gpu_str = " | ".join(gpu_stats) if gpu_stats else "CPU"
            print(f"  [Resource Step {step:5d}] RAM: {ram_gb:.2f} GB | GPU Mem: {gpu_str}")
            sys.stdout.flush()
        
        # Validation evaluation every VAL_EVERY (500) steps (or at end of SMOKE run) (Task 3)
        if (step % VAL_EVERY == 0) or (SMOKE and step == target_steps):
            adapter.eval()
            bare_net = model.module if isinstance(model, nn.DataParallel) else model
            bare_net.eval()
            cur_scores = []
            with torch.no_grad():
                for item in val_fixed_clips:
                    c_np = item['clean']
                    ny_np = item['noisy']
                    df_state.reset()
                    enh_cur = enhance(bare_net, df_state, torch.from_numpy(ny_np).unsqueeze(0)).squeeze().cpu().numpy()
                    sc = calc_metrics_for_eval(c_np, enh_cur)
                    cur_scores.append(sc)
                    item['cur_sisdr'] = sc[1]
                
            c_mean = np.nanmean(cur_scores, axis=0)
        
            # Per-stratum SI-SDR
            cur_strata_sisdr = {}
            for s_name in strata_names:
                s_vals = [item['cur_sisdr'] for item in val_fixed_clips if item['stratum'] == s_name]
                cur_strata_sisdr[s_name] = float(np.nanmean(s_vals))
            
            # Gunfire-only SI-SDR across the 38 gunfire clips
            cur_gun_sisdr = float(np.nanmean([item['cur_sisdr'] for item in val_fixed_clips if item['is_gunfire']]))
        
            # V3.4: Score = overall SI-SDR + PESQ + 0.5*gunfire SI-SDR (drives burst suppression)
            val_score = c_mean[1] + (c_mean[3] if not np.isnan(c_mean[3]) else 0.0) + 0.5 * cur_gun_sisdr
        
            # Side-by-side validation printing against precomputed baseline (Task 3)
            print(f"\n--> [VAL Step {step}] Model: SNR={c_mean[0]:.2f} | SI-SDR={c_mean[1]:.2f} | STOI={c_mean[2]:.3f} | PESQ={c_mean[3]:.2f} | Lvl={c_mean[4]:+.2f}dB")
            print(f"    [VAL Step {step}] Pretr: SNR={pre_mean[0]:.2f} | SI-SDR={pre_mean[1]:.2f} | STOI={pre_mean[2]:.3f} | PESQ={pre_mean[3]:.2f} | Lvl={pre_mean[4]:+.2f}dB")
            print(f"    [VAL Step {step}] Gunfire-Only SI-SDR (38 clips): Model = {cur_gun_sisdr:.2f} dB | Pretrained = {pre_gun_sisdr:.2f} dB (diff: {cur_gun_sisdr - pre_gun_sisdr:+.2f} dB)")
            print(f"    [VAL Step {step}] Per-Stratum SI-SDR Breakdown (Model vs Pretrained):")
            strata_summary = " ".join([f"{s_name}:{cur_strata_sisdr[s_name]:.1f}/{pre_strata_sisdr[s_name]:.1f}" for s_name in strata_names])
            print(f"      {strata_summary}\n")
        
            # Append to validation_log.csv
            log_row = [
                step, f"{lr:.2e}", f"{loss_val:.4f}", f"{float(l_comp.detach()):.4f}", f"{float(l_mag.detach()):.4f}", f"{float(l_sdr.detach()):.2f}",
                f"{c_mean[0]:.2f}", f"{c_mean[1]:.2f}", f"{c_mean[2]:.3f}", f"{c_mean[3]:.2f}", f"{c_mean[4]:.2f}",
                f"{pre_mean[0]:.2f}", f"{pre_mean[1]:.2f}", f"{pre_mean[2]:.3f}", f"{pre_mean[3]:.2f}", f"{pre_mean[4]:.2f}",
                f"{cur_gun_sisdr:.2f}", f"{pre_gun_sisdr:.2f}"
            ] + [f"{cur_strata_sisdr[s_name]:.2f}" for s_name in strata_names]
        
            with open(val_log_file, 'a', newline='') as f:
                csv.writer(f).writerow(log_row)
            
            # Best model checkpoint save (saves both .hk and .pt by validation SI-SDR + PESQ)
            if val_score > best_val_score:
                best_val_score = val_score
                save_dual_checkpoint(
                    model, OUT_DIR / 'best',
                    stage_name='best_checkpoint',
                    metadata={'step': str(step), 'sisdr': f"{c_mean[1]:.2f}", 'pesq': f"{c_mean[3]:.2f}"}
                )
                print(f"*** New BEST checkpoint saved: best.hk & best.pt (Val Score = {best_val_score:.2f}) ***")
            
            save_dual_checkpoint(model, OUT_DIR / 'last', stage_name='last_checkpoint', metadata={'step': str(step)})
            adapter.train()

        # Checkpoint saving every SAVE_EVERY (500) steps (Task 3: permanent every 2000 steps)
        if step % SAVE_EVERY == 0:
            step_base = CKPT_DIR / f'step-{step}'
            save_dual_checkpoint(model, step_base, stage_name=f'step_{step}', metadata={'step': str(step)})
        
            # Task 3: Keep EVERY 2000-step checkpoint permanently (.pt and .hk)
            if step % 2000 == 0:
                print(f"*** Permanent Milestone Checkpoint Saved: step-{step}.hk & step-{step}.pt (permanent retention) ***")
            else:
                saved_rolling_ckpts.append(step_base)
                while len(saved_rolling_ckpts) > MAX_KEEP_CKPT:
                    old_base = saved_rolling_ckpts.pop(0)
                    for ext in ['.hk', '.pt']:
                        p_old = old_base.with_suffix(ext)
                        if p_old.exists():
                            p_old.unlink()
                            print(f"Evicted old rolling checkpoint: {p_old.name} (retaining last {MAX_KEEP_CKPT})")
except BaseException as e:
    import traceback
    err_tb = traceback.format_exc()
    err_file = Path('/kaggle/working/error.txt')
    try:
        with open(err_file, 'w') as ef:
            ef.write(err_tb)
        print('[CRASH LOG] Wrote complete traceback to /kaggle/working/error.txt')
    except Exception as io_err:
        print('[CRASH LOG] Failed to write error file:', io_err)
    print('='*70)
    print('[CRASH LOG] CRITICAL EXCEPTION HALTED TRAINING AT STEP:', step)
    print(err_tb)
    print('='*70)
    try:
        bare_net_err = model.module if isinstance(model, nn.DataParallel) else model
        torch.save(bare_net_err.state_dict(), '/kaggle/working/last_on_error.pt')
        print('[CRASH LOG] Successfully preserved current bare model weights to /kaggle/working/last_on_error.pt')
    except Exception as save_err:
        print('[CRASH LOG] Failed to save last_on_error.pt:', save_err)
    sys.stdout.flush()
    sys.stderr.flush()
    raise

    # Fix 8: Profile and stop if SMOKE mode
if SMOKE:
    gpu_mem_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
    mean_step_time = np.mean(step_times)
    print(f"\n=== SMOKE TEST RUN COMPLETE ===")
    print(f"Steps Executed     : {step}")
    print(f"Mean Step Latency  : {mean_step_time*1000:.1f} ms/step ({1.0/max(mean_step_time, 1e-6):.2f} steps/s)")
    print(f"Peak GPU Memory    : {gpu_mem_mb:.1f} MB")
    print(f"Smoke validation complete. Exiting cleanly as requested.")
else:
    print("\nOptimization complete! Finalizing artifacts...")
    save_dual_checkpoint(model, OUT_DIR / 'final_model', stage_name='final_model', metadata={'steps': str(step)})
    print(f"All training artifacts saved in {OUT_DIR}")
